In [1]:
from cell_labelling_functions import *

In [2]:
######### INPUTS ###########
# path to CODEX stardist segmented geojson
# pthgeojson=r'\\169.254.138.20\Andre\data\Ashleigh fallopian tube\fallopian tubes\CODEX\dk_stardist\AJER376 R_090524_Scan1_Fimbriated_only.qptiff - resolution #1.geojson'
pthgeojson=r'\\169.254.138.20\Andre\data\Ashleigh fallopian tube\fallopian tubes\CODEX\dk_stardist\AJER376 R_090524_Scan1.qptiff - resolution #1.geojson'

# path to output folder where the CODEX geojson with the clusters will be saved
outpth=r'\\169.254.138.20\Andre\data\Ashleigh fallopian tube\fallopian tubes\CODEX\dk_stardist'

# path to the centroids file (output of the clustering analysis)
# pthcentroidscluster=r'\\10.99.68.178\Saurabh\Andre_Cell2Class\matrix_clustered'
# pthcentroidscluster=r'\\169.254.138.20\Andre\data\Ashleigh fallopian tube\fallopian tubes\CODEX\Saurabh_clustering_analysis\matrix_clustered_no_PCNA_res1'
# pthcentroidscluster=r'\\10.99.68.178\Saurabh\Andre_Cell2Class\combined_cluster_plot_wholeFT_no_PCNA_res_1_fx_cl_v3_FINAL'
pthcentroidscluster=r'\\10.99.68.178\Saurabh\Andre_Cell2Class\combined_cluster_plot_wholeFT_no_PCNA_res_1_fx_cl_v3_FINALv2'

############################
outpthlabels = os.path.join(outpth, 'cell_labelling_data')

# Create directories if they don't exist
for path in [outpthlabels]:
    os.makedirs(path, exist_ok=True)

In [3]:
# read geojson file
df = gpd.read_file(pthgeojson)

In [4]:
# Create a dataframe with the extracted measurements
measurements_list = df.apply(extract_measurements, axis=1)
measurements_df = pd.json_normalize(measurements_list)

# Combine the measurements dataframe with the original GeoDataFrame
final_df = pd.concat([df[['id', 'objectType', 'geometry']], measurements_df], axis=1)
final_df_cleaned = final_df.dropna(thresh=final_df.shape[1] // 2)
# Display the cleaned dataframe
# print(final_df_cleaned.head(3))

# Save the cleaned dataframe to a CSV file
# final_df_cleaned.head(5).to_csv(os.path.join(outpthlabels,'cleaned_geojson_measurements.csv'), index=False)

In [5]:
# Step 6: Load the centroid data (replace 'centroids_file.csv' with your file path)
# centroids_df = pd.read_csv(os.path.join(pthcentroidscluster,'clustered_data.csv'))
# centroids_df = pd.read_csv(os.path.join(pthcentroidscluster,'clustered_data_cell_phenotypes.csv'))
centroids_df = pd.read_csv(os.path.join(pthcentroidscluster,'normalized_expression_matrix_STIC.csv'))
centroids_df.head(3)

,Cell_index,Centroid_x,Centroid_y,Cell_area,CD4,IFNG,CD20,CD45RO,Pan-CK,CD141,...,EpCAM,IDO1,MPO,Podoplanin,HLA-DR,CD45,CD31,CD44,Cluster,color
0,0,12961.43,1232.98,223.03,0.383780,0.545859,-1.932395,-0.89032,1.161985,1.384952,...,-1.028605,0.512487,-0.135819,0.785239,0.019697,0.215532,-1.04543,-0.335520,Dendritic Cells,#E377C2
1,1,12908.08,1251.93,252.02,-1.637483,1.299697,-0.611010,-0.89032,-1.182670,1.428707,...,0.921909,-1.067835,0.384328,1.586283,-1.416715,-0.581719,-1.04543,0.886694,Fibroblasts / Mesenchymal Cells,#AEC7E8
2,2,12959.36,1268.01,140.18,-0.608492,2.280041,-0.897440,-0.89032,-0.589392,0.304118,...,1.076116,-0.527556,0.390600,2.354355,-0.830123,-0.026414,-1.04543,0.867566,Fibroblasts / Mesenchymal Cells,#AEC7E8


In [6]:
# Step 7: Extract the 'Cluster' column to get the cluster IDs
cluster_ids = centroids_df['Cluster'].values
centroids = centroids_df[['Centroid_x', 'Centroid_y']].values
# print(centroids[:5])
# print(cluster_ids[:5])

In [7]:
# Step 8: Extract coordinates of geometries as a 2D array (getting centroids of polygons)
geo_coords = pd.DataFrame(df['geometry'].apply(lambda x: (x.representative_point().x, x.representative_point().y)).to_list()).values
print(geo_coords[:5])

[[17760.         25531.56      ]
 [13056.59891967   137.11      ]
 [12961.52622037  1232.495     ]
 [12908.03867043  1252.505     ]
 [12960.47190409  1267.4       ]]


In [8]:
# Step 9: Use Nearest Neighbors for finding the closest centroids
# Using a spatial index for more efficient nearest neighbor search
nbrs = NearestNeighbors(n_neighbors=1, algorithm='auto').fit(centroids)

# Step 10: Find the nearest centroid for each geometry
distances, indices = nbrs.kneighbors(geo_coords)

In [9]:
# Step 11: Assign the nearest cluster ID to each geometry
df['Cluster'] = cluster_ids[indices.flatten()]  # Assign the nearest cluster ID
# df.head(3)

In [10]:
# Step 12: Assign colors based on cluster IDs
# cmap = plt.get_cmap('tab20')
# norm = mcolors.Normalize(vmin=cluster_ids.min(), vmax=cluster_ids.max())
# df['color'] = df['Cluster'].apply(lambda x: mcolors.rgb2hex(cmap(norm(x))))
# df.head(3)

ValueError: could not convert string to float: 'Activated T Cells'

In [10]:
# Step: Remove measurement columns to reduce file size
columns_to_drop = [col for col in df.columns if 'measurements' in col]
df_no_measurements = df.drop(columns=columns_to_drop)
# Save the updated GeoJSON without the measurements data
df_no_measurements.head(3)

,id,objectType,isLocked,geometry,Cluster
0,65b5795d-79d6-4997-bfec-a9057374b1dc,annotation,1.0,"POLYGON ((8914.91 0, 8845.71 36.96, 8805 49, 8...",B Cells
1,4046894d-0eee-40f2-b866-5f937d9c4c82,cell,NaN,"POLYGON ((13059.19 123.53, 13051.49 124.39, 13...",CD4 memory helper T cell
2,8060fc19-d86b-40a4-b44e-d5926eb90de2,cell,NaN,"POLYGON ((12963.49 1215.96, 12958.4 1215.97, 1...",Dendritic Cells


In [11]:
# df_no_measurements.to_csv(os.path.join(outpthlabels, 'df_no_measurements.csv'), index=False)
df_no_measurements.to_csv(os.path.join(outpthlabels, 'df_no_measurement_STIC.csv'), index=False)

In [ ]:
# read the saved csv file
df_no_measurements = pd.read_csv(os.path.join(outpthlabels, 'df_no_measurement_STIC.csv'))


In [12]:
phenotype_colors = {
    "Activated T Cells": "#1f77b4",  # Blue
    "Antigen-Presenting Cells (APCs)": "#ff7f0e",  # Orange
    "B Cells": "#2ca02c",  # Green
    "CD4 memory helper T cell": "#d62728",  # Red
    "CD8 memory cytotoxic T cell": "#9467bd",  # Purple
    "CD8+ T cells": "#8c564b",  # Brown
    "Dendritic Cells": "#e377c2",  # Pink
    "Endothelial Cells": "#bcbd22",  # Olive
    "Epithelial Cells": "#17becf",  # Cyan
    "Fibroblasts / Mesenchymal Cells": "#aec7e8",  # Light Blue
    "Macrophages /Monocytes": "#ffbb78",  # Peach
    "Myofibroblasts": "#98df8a",  # Light Green
    "Neutrophils": "#ff9896",  # Light Red
    "Proliferating Cells (Ki67)": "#c5b0d5",  # Lavender
    "Regulatory Dendritic Cells (DCs)": "#c49c94",  # Tan
    "STIC": "#FF0000",  # red
    "Smooth muscle Cells": "#dbdb8d",  # Pale Yellow
    "Tumor-Associated Macrophages": "#9edae5",  # Light Cyan
    "suppressed dendritic": "#8c564b",  # Same as CD8+ T cells (Dark Brown)
}

In [13]:
# Get the unique cluster IDs and sort them
unique_clusters = sorted(df_no_measurements['Cluster'].unique())
# Create class names in the format 'Cluster 1', 'Cluster 2', etc.
# class_names = [f'Cluster {int(cluster)}' for cluster in unique_clusters]
class_names = [f'{cell_type}' for cell_type in unique_clusters]
print(class_names)

['Activated T Cells', 'Antigen-Presenting Cells (APCs)', 'B Cells', 'CD4 memory helper T cell', 'CD8 memory cytotoxic T cell', 'CD8+ T cells', 'Dendritic Cells', 'Endothelial Cells', 'Epithelial Cells', 'Fibroblasts / Mesenchymal Cells', 'Macrophages /Monocytes', 'Myofibroblasts', 'Neutrophils', 'Proliferating Cells (Ki67)', 'Regulatory Dendritic Cells (DCs)', 'STIC', 'Smooth muscle Cells', 'Tumor-Associated Macrophages', 'suppressed dendritic']


In [20]:
def save_GEOJSON_with_phenotypes(out_pth_json, df, phenotype_names, phenotype_colors_dict=None):
    """
    Save GeoJSON with cell phenotypes, class names, and assigned colors.

    Args:
        out_pth_json: Output path for the GeoJSON file
        df: DataFrame containing the cell data with 'Cluster' and 'geometry' columns
        phenotype_names: List of phenotype names to include
        phenotype_colors_dict: Dictionary mapping phenotype names to hex color strings
    """
    # Ensure phenotype_names is provided
    if phenotype_names is None:
        raise ValueError("phenotype_names must be provided.")

    # Create a GeoJSON dictionary
    GEOdata = []

    for index, row in df.iterrows():
        phenotype = row['Cluster']
        contour = row['geometry']

        # Ensure the phenotype is valid and exists in phenotype_names
        if phenotype in phenotype_names:
            # Get the color for this phenotype
            if phenotype_colors_dict and phenotype in phenotype_colors_dict:
                # Convert hex color to RGB
                hex_color = phenotype_colors_dict[phenotype]
                phenotype_color = [int(c * 255) for c in mcolors.hex2color(hex_color)]
            else:
                # Default color if not specified
                phenotype_color = [128, 128, 128]  # Gray

            # Extract coordinates from contour, handling both Polygon and MultiPolygon geometries
            if isinstance(contour, Polygon):
                # Convert to list and ensure closed polygon
                exterior = list(contour.exterior.coords)
                if exterior[0] != exterior[-1]:
                    exterior.append(exterior[0])
                # Format coordinates properly for GeoJSON
                coordinates = [[[float(round(coord, 2)) for coord in point] for point in exterior]]
            elif isinstance(contour, MultiPolygon):
                coordinates = []
                for polygon in contour.geoms:
                    exterior = list(polygon.exterior.coords)
                    if exterior[0] != exterior[-1]:
                        exterior.append(exterior[0])
                    # Format coordinates properly for GeoJSON
                    coordinates.append([[[float(round(coord, 2)) for coord in point] for point in exterior]])
            else:
                continue  # Skip invalid geometry types

            # Create GeoJSON feature
            dict_data = {
                "type": "Feature",
                "id": f"PathCellObject_{index}",
                "geometry": {
                    "type": "Polygon" if isinstance(contour, Polygon) else "MultiPolygon",
                    "coordinates": coordinates
                },
                "properties": {
                    'objectType': 'annotation',
                    'classification': {
                        'name': str(phenotype),
                        'color': phenotype_color
                    }
                }
            }

            GEOdata.append(dict_data)

    # Create FeatureCollection
    geojson_dict = {
        "type": "FeatureCollection",
        "features": GEOdata
    }

    # Save GeoJSON to file
    geojson_path = os.path.join(out_pth_json, 'updated_segmentation_STIC.geojson')
    with open(geojson_path, 'w') as outfile:
        geojson.dump(geojson_dict, outfile)
    print('GeoJSON file saved to', geojson_path)

In [21]:
# save_GEOJSON_with_clusters(outpthlabels, df_no_measurements, class_names)
# save_GEOJSON_with_phenotypes(outpthlabels, df_no_measurements, class_names)
# Call the function with your phenotype colors
save_GEOJSON_with_phenotypes(outpthlabels, df_no_measurements, class_names, phenotype_colors)

GeoJSON file saved to \\169.254.138.20\Andre\data\Ashleigh fallopian tube\fallopian tubes\CODEX\dk_stardist\cell_labelling_data\updated_segmentation_STIC.geojson


In [ ]:
# import colorsys
# import distinctipy
#
# # Assume class_names is your list of 17 cell types
# num_celltypes = len(class_names)
#
# # Generate 17 highly distinct colors
# colors = distinctipy.get_colors(num_celltypes)
#
# # Create a mapping from each cell type to its color
# celltype_color_map = dict(zip(class_names, colors))
#
# # Plot each cell type with its assigned color
# plt.figure(figsize=(8, 6))
# for i, (cell_type, color) in enumerate(celltype_color_map.items()):
#     # Plot a colored circle for each cell type
#     plt.scatter(0, i, s=300, color=color, edgecolors='black')
#     # Annotate with the cell type name
#     plt.text(0.1, i, cell_type, fontsize=12, verticalalignment='center')
#
# # Clean up the plot
# plt.xlim(-0.2, 3)
# plt.ylim(-1, num_celltypes)
# plt.axis('off')
# plt.title("Highly Distinct Colors for Each Cell Type", fontsize=16)
# plt.show()